# 02 - Feature Engineering

Ce notebook développe les features pour le modèle de prédiction du risque vaccinal.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully")

## 1. Chargement et préparation des données

In [ ]:
# Charger les données (simulées pour l'instant)
def generate_enhanced_data(n_enfants=1000):
    """Génère des données plus réalistes pour le feature engineering"""
    np.random.seed(42)
    
    data = {
        'enfant_id': [f'enfant_{i:04d}' for i in range(n_enfants)],
        'date_naissance': [datetime.now() - timedelta(days=np.random.randint(30, 1825)) for _ in range(n_enfants)],
        'sexe': np.random.choice(['M', 'F'], n_enfants),
        'region': np.random.choice(['Dakar', 'Thiès', 'Kaolack', 'Saint-Louis', 'Diourbel'], n_enfants),
        'centre_sante': [f'Centre_{np.random.randint(1, 11)}' for _ in range(n_enfants)],
        'nombre_vaccinations': np.random.poisson(3, n_enfants),
        'derniere_vaccination': [datetime.now() - timedelta(days=np.random.randint(1, 365)) for _ in range(n_enfants)],
        'retard_vaccinal': np.random.choice([0, 1], n_enfants, p=[0.7, 0.3]),
        'nombre_tuteurs': np.random.randint(1, 4, n_enfants),
        'distance_centre_km': np.random.exponential(5, n_enfants),
        'saison_naissance': np.random.choice(['Sèche', 'Pluvieuse'], n_enfants)
    }
    
    return pd.DataFrame(data)

df = generate_enhanced_data()
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Features temporelles

In [ ]:
def create_temporal_features(df):
    """Crée des features temporelles"""
    df = df.copy()
    
    # Features d'âge
    df['age_jours'] = (datetime.now() - df['date_naissance']).dt.days
    df['age_mois'] = df['age_jours'] // 30
    df['age_annees'] = df['age_jours'] / 365.25
    
    # Features de vaccination temporelles
    df['jours_derniere_vaccination'] = (datetime.now() - df['derniere_vaccination']).dt.days
    df['mois_derniere_vaccination'] = df['jours_derniere_vaccination'] // 30
    
    # Features saisonnières
    df['mois_naissance'] = df['date_naissance'].dt.month
    df['saison_naissance_num'] = df['saison_naissance'].map({'Sèche': 0, 'Pluvieuse': 1})
    
    # Ratios temporels
    df['vaccinations_par_mois'] = df['nombre_vaccinations'] / (df['age_mois'] + 1)
    df['intervalle_moyen_vaccinations'] = df['jours_derniere_vaccination'] / (df['nombre_vaccinations'] + 1)
    
    return df

df_temporal = create_temporal_features(df)
print("Features temporelles créées")
df_temporal[['age_mois', 'jours_derniere_vaccination', 'vaccinations_par_mois']].head()

## 3. Features géographiques

In [ ]:
def create_geographic_features(df):
    """Crée des features géographiques"""
    df = df.copy()
    
    # Encodage des régions
    region_encoder = LabelEncoder()
    df['region_encoded'] = region_encoder.fit_transform(df['region'])
    
    # Features de distance
    df['distance_centre_log'] = np.log1p(df['distance_centre_km'])
    df['distance_categorie'] = pd.cut(df['distance_centre_km'], 
                                    bins=[0, 2, 5, 10, np.inf], 
                                    labels=['Proche', 'Moyenne', 'Loin', 'Très loin'])
    
    # Encodage des catégories de distance
    distance_encoder = LabelEncoder()
    df['distance_categorie_encoded'] = distance_encoder.fit_transform(df['distance_categorie'])
    
    return df

df_geo = create_geographic_features(df_temporal)
print("Features géographiques créées")
df_geo[['region_encoded', 'distance_centre_log', 'distance_categorie_encoded']].head()

## 4. Features démographiques

In [ ]:
def create_demographic_features(df):
    """Crée des features démographiques"""
    df = df.copy()
    
    # Encodage du sexe
    df['sexe_encoded'] = (df['sexe'] == 'M').astype(int)
    
    # Features familiales
    df['log_nombre_tuteurs'] = np.log1p(df['nombre_tuteurs'])
    df['ratio_tuteurs_enfant'] = df['nombre_tuteurs'] / 1  # Normalisé pour 1 enfant
    
    # Features d'interaction
    df['sexe_x_age'] = df['sexe_encoded'] * df['age_mois']
    df['sexe_x_distance'] = df['sexe_encoded'] * df['distance_centre_log']
    
    return df

df_demo = create_demographic_features(df_geo)
print("Features démographiques créées")
df_demo[['sexe_encoded', 'log_nombre_tuteurs', 'sexe_x_age']].head()

## 5. Features de comportement vaccinal

In [ ]:
def create_vaccination_behavior_features(df):
    """Crée des features liées au comportement vaccinal"""
    df = df.copy()
    
    # Scores de couverture
    df['score_couverture_age'] = df['nombre_vaccinations'] / (df['age_mois'] / 6 + 1)  # 6 mois = 1 vaccin attendu
    df['score_regularite'] = 1 / (df['intervalle_moyen_vaccinations'] + 1)
    
    # Features de risque
    df['risque_age_inapproprie'] = (df['age_mois'] < 6) & (df['nombre_vaccinations'] == 0).astype(int)
    df['risque_intervalle_long'] = (df['intervalle_moyen_vaccinations'] > 90).astype(int)
    df['risque_distance_elevee'] = (df['distance_centre_km'] > 10).astype(int)
    
    # Features composites
    df['score_risque_global'] = (
        df['risque_age_inapproprie'] + 
        df['risque_intervalle_long'] + 
        df['risque_distance_elevee']
    ) / 3
    
    return df

df_final = create_vaccination_behavior_features(df_demo)
print("Features de comportement créées")
df_final[['score_couverture_age', 'score_regularite', 'score_risque_global']].head()

## 6. Analyse des features créées

In [ ]:
# Sélection des features numériques pour l'analyse
numeric_features = [
    'age_mois', 'jours_derniere_vaccination', 'vaccinations_par_mois',
    'region_encoded', 'distance_centre_log', 'sexe_encoded',
    'log_nombre_tuteurs', 'sexe_x_age', 'score_couverture_age',
    'score_regularite', 'score_risque_global', 'retard_vaccinal'
]

df_features = df_final[numeric_features]

# Matrice de corrélation
plt.figure(figsize=(12, 8))
correlation_matrix = df_features.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f')
plt.title('Matrice de corrélation des features')
plt.tight_layout()
plt.show()

In [ ]:
# Importance des features (corrélation avec la cible)
feature_importance = correlation_matrix['retard_vaccinal'].abs().sort_values(ascending=False)
feature_importance = feature_importance[feature_importance.index != 'retard_vaccinal']

plt.figure(figsize=(10, 6))
feature_importance.plot(kind='bar')
plt.title('Importance des features (corrélation absolue avec retard_vaccinal)')
plt.xlabel('Features')
plt.ylabel('Corrélation absolue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nTop 10 des features les plus importantes:")
print(feature_importance.head(10))

## 7. Préparation pour le modèle

In [ ]:
# Préparation des données pour le modèle
def prepare_model_data(df):
    """Prépare les données finales pour le modèle"""
    
    # Sélection des features finales
    final_features = [
        'age_mois', 'jours_derniere_vaccination', 'vaccinations_par_mois',
        'region_encoded', 'distance_centre_log', 'sexe_encoded',
        'log_nombre_tuteurs', 'score_couverture_age', 'score_regularite',
        'score_risque_global'
    ]
    
    X = df[final_features]
    y = df['retard_vaccinal']
    
    # Standardisation
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    
    return X_scaled, y, scaler

X, y, scaler = prepare_model_data(df_final)

print(f"Shape des features: {X.shape}")
print(f"Distribution de la cible: {y.value_counts(normalize=True)}")
print(f"\nFeatures finales: {list(X.columns)}")

In [ ]:
# Sauvegarde des données préparées
df_final.to_csv('../data/prepared_features.csv', index=False)
X.to_csv('../data/features_scaled.csv', index=False)
y.to_csv('../data/target.csv', index=False)

# Sauvegarde du scaler
import joblib
joblib.dump(scaler, '../models/feature_scaler.pkl')

print("Données préparées et sauvegardées avec succès!")

## 8. Résumé du Feature Engineering

### Features créées:

#### 1. Features temporelles:
- `age_mois`: Âge en mois
- `jours_derniere_vaccination`: Jours depuis la dernière vaccination
- `vaccinations_par_mois`: Ratio vaccinations/âge
- `intervalle_moyen_vaccinations`: Intervalle moyen entre vaccinations

#### 2. Features géographiques:
- `region_encoded`: Région encodée
- `distance_centre_log`: Distance log-transformée
- `distance_categorie_encoded`: Catégorie de distance

#### 3. Features démographiques:
- `sexe_encoded`: Sexe encodé
- `log_nombre_tuteurs`: Nombre de tuteurs (log)
- `sexe_x_age`: Interaction sexe*âge

#### 4. Features de comportement:
- `score_couverture_age`: Score de couverture vaccinale
- `score_regularite`: Score de régularité
- `score_risque_global`: Score de risque composite

### Prochaines étapes:
1. Entraînement du modèle avec ces features
2. Validation croisée
3. Optimisation des hyperparamètres
4. Interprétabilité du modèle